# Conflict Monitor Extraction

This is a prompt to give an LLM to extract conflict pairs from a screenshot of the conflict monitor from the cabinet print.

```

I have attached an image of a Conflict Monitor Diode Card for a traffic signal. I need you to write a Python script that will extract the conflict pairs and append them to my conflict_pairs.json file. I have attached multiple conflict monitors, with each image named by the signal. I want you to update the script to add all of them, with a single copy/paste script, add all at once.

How to interpret the card:
1. Channel Assignment (Top): Maps Channels (1-16) to movements. 
- Phases (e.g., "Ø1") map to "Ph1", "Ph2", etc.
- Overlaps (e.g., "OLA", "OLB", "OLC", "OLD" or "FYA") map to "O1", "O2", "O3", "O4" (A=1, B=2, C=3, D=4).
- Peds (e.g., "PED 2") map to "Ped2", "Ped4", etc.
- "N/U" maps to None.
2. Diodes (Bottom): Shows the matrix. 
- Empty half-circle = Diode REMOVED (Allowable / No Conflict).
- Solid black bar = Diode PRESENT (Conflict).

Your Task:
Write a Python script using the template below. You only need to fill in the signal_id, the channels dictionary based on the top of the card, and the removed_diodes list by carefully identifying every empty half-circle in the matrix.

--- SCRIPT TEMPLATE ---

import json
from pathlib import Path

signal_id = "INSERT_SIGNAL_ID_HERE"

# 1. Fill this out based on the Channel Assignment section
channels = {
    1: 'Ph1',
    2: 'Ph2',
    3: None,
    4: 'Ph4',
    5: 'Ph5',
    6: 'Ph6',
    7: None,
    8: 'Ph8',
    9: 'O1',
    10: 'O2',
    11: 'O3',
    12: 'O4',
    13: 'Ped2',
    14: 'Ped4',
    15: 'Ped6',
    16: 'Ped8'
}

# 2. List all pairs that have the diode REMOVED (the empty half-circles)
# Example: (1, 5), (2, 6)
removed_diodes = [
    # ... fill in all empty half-circles here
]

# --- DO NOT CHANGE THE LOGIC BELOW ---
active_channels = [ch for ch, name in channels.items() if name is not None]
all_pairs = [(min(c1, c2), max(c1, c2)) for i, c1 in enumerate(active_channels) for c2 in active_channels[i+1:]]
conflict_pairs = [[channels[p[0]], channels[p[1]]] for p in all_pairs if p not in removed_diodes]

json_path = Path('version_testing/conflict_monitor/conflict_pairs.json')
data = json.loads(json_path.read_text()) if json_path.exists() else {}
data[signal_id] = conflict_pairs

# Custom formatting for single-line pairs
json_str = '{\n'
items = list(data.items())
for i, (sig, pairs) in enumerate(items):
    json_str += f'    "{sig}": [\n'
    json_str += ',\n'.join([f'        ["{p[0]}", "{p[1]}"]' for p in pairs])
    json_str += '\n    ]' + (',' if i < len(items) - 1 else '') + '\n'
json_str += '}\n'

json_path.write_text(json_str)
print(f"Added {len(conflict_pairs)} conflict pairs for signal {signal_id}")

```

In [2]:
import json
from pathlib import Path

# Data for all 8 Conflict Monitor Cards
signals_data = [
    {
        "signal_id": "12059",
        "channels": {
            1: 'Ph1', 2: 'Ph2', 3: None, 4: 'Ph4',
            5: 'Ph5', 6: 'Ph6', 7: None, 8: 'Ph8',
            9: 'O1', 10: 'O2', 11: 'O3', 12: 'O4',
            13: 'Ped2', 14: 'Ped4', 15: 'Ped6', 16: 'Ped8'
        },
        "removed_diodes": [
            (1, 5), (1, 6), (1, 11),
            (2, 5), (2, 6), (2, 9), (2, 10), (2, 12), (2, 13), (2, 15),
            (4, 10), (4, 11), (4, 14),
            (5, 9), (5, 10), (5, 11), (5, 12), (5, 13),
            (6, 9), (6, 10), (6, 11), (6, 13), (6, 15),
            (8, 9), (8, 16),
            (9, 10), (9, 13), (9, 15), (9, 16),
            (10, 11), (10, 12), (10, 13), (10, 14), (10, 15),
            (11, 14), (11, 15),
            (12, 13), (12, 15),
            (13, 15)
        ]
    },
    {
        "signal_id": "13008",
        "channels": {
            1: 'Ph1', 2: 'Ph2', 3: None, 4: 'Ph4',
            5: 'Ph5', 6: 'Ph6', 7: None, 8: 'Ph8',
            9: None, 10: None, 11: None, 12: None,
            13: 'Ped2', 14: 'Ped4', 15: 'Ped6', 16: None
        },
        "removed_diodes": [
            (1, 5), (1, 6), (1, 15),
            (2, 5), (2, 6), (2, 13), (2, 15),
            (4, 14),
            (5, 13),
            (6, 13), (6, 15),
            (13, 15)
        ]
    },
    {
        "signal_id": "13010",
        "channels": {
            1: 'Ph1', 2: 'Ph2', 3: 'Ph3', 4: 'Ph4',
            5: 'Ph5', 6: 'Ph6', 7: 'Ph7', 8: 'Ph8',
            9: 'O1', 10: 'O3', 11: 'O5', 12: None,
            13: 'Ped2', 14: 'Ped4', 15: 'Ped6', 16: 'Ped8'
        },
        "removed_diodes": [
            (1, 5), (1, 6), (1, 11), (1, 15),
            (2, 5), (2, 6), (2, 9), (2, 13), (2, 15),
            (3, 7), (3, 8), (3, 14),
            (4, 7), (4, 8), (4, 10), (4, 14), (4, 16),
            (5, 9), (5, 13),
            (6, 9), (6, 11), (6, 13), (6, 15),
            (7, 14),
            (8, 10), (8, 14), (8, 16),
            (9, 13), (9, 15),
            (10, 14), (10, 16),
            (11, 13), (11, 15),
            (13, 15),
            (14, 16)
        ]
    },
    {
        "signal_id": "2C043",
        "channels": {
            1: 'Ph1', 2: 'Ph2', 3: 'O1', 4: 'Ph4',
            5: 'Ph5', 6: 'Ph6', 7: None, 8: 'Ph8',
            9: None, 10: None, 11: None, 12: None,
            13: 'Ped2', 14: 'Ped4', 15: 'Ped6', 16: 'Ped8'
        },
        "removed_diodes": [
            (1, 3), (1, 5), (1, 6),
            (2, 3), (2, 5), (2, 6), (2, 13), (2, 15),
            (3, 4), (3, 5), (3, 8), (3, 13), (3, 14), (3, 16),
            (4, 8), (4, 14), (4, 16),
            (5, 13),
            (6, 13), (6, 15),
            (8, 14), (8, 16),
            (13, 15),
            (14, 16)
        ]
    },
    {
        "signal_id": "2B049",
        "channels": {
            1: 'Ph1', 2: 'Ph2', 3: None, 4: 'Ph4',
            5: 'Ph5', 6: 'Ph6', 7: None, 8: 'Ph8',
            9: None, 10: None, 11: None, 12: None,
            13: 'Ped2', 14: 'Ped4', 15: 'Ped6', 16: 'Ped8'
        },
        "removed_diodes": [
            (1, 5), (1, 6), (1, 15),
            (2, 5), (2, 6), (2, 13), (2, 15),
            (4, 8), (4, 14), (4, 16),
            (5, 13),
            (6, 13), (6, 15),
            (8, 14), (8, 16),
            (13, 15),
            (14, 16)
        ]
    },
    {
        "signal_id": "2B052",
        "channels": {
            1: 'Ph1', 2: 'Ph2', 3: 'Ph3', 4: 'Ph4',
            5: 'Ph5', 6: 'Ph6', 7: 'Ph7', 8: 'Ph8',
            9: 'O1', 10: None, 11: None, 12: None,
            13: 'Ped2', 14: 'Ped4', 15: 'Ped6', 16: 'Ped8'
        },
        "removed_diodes": [
            (1, 5), (1, 6), (1, 9), (1, 15),
            (2, 5), (2, 6), (2, 13), (2, 15),
            (3, 7), (3, 8), (3, 14), (3, 16),
            (4, 7), (4, 8), (4, 14), (4, 16),
            (5, 9), (5, 13),
            (6, 9), (6, 13), (6, 15),
            (7, 14), (7, 16),
            (8, 9), (8, 14), (8, 16),
            (9, 13), (9, 16),
            (13, 15),
            (14, 16)
        ]
    },
    {
        "signal_id": "2B094",
        "channels": {
            1: 'Ph1', 2: 'Ph2', 3: None, 4: 'Ph4',
            5: 'Ph5', 6: 'Ph6', 7: None, 8: 'Ph8',
            9: 'O1', 10: None, 11: 'O5', 12: None,
            13: 'Ped2', 14: 'Ped4', 15: 'Ped6', 16: 'Ped8'
        },
        "removed_diodes": [
            (1, 5), (1, 6), (1, 11), (1, 15),
            (2, 5), (2, 6), (2, 9), (2, 13), (2, 15),
            (4, 8), (4, 14), (4, 16),
            (5, 9), (5, 13),
            (6, 9), (6, 11), (6, 13), (6, 15),
            (8, 14), (8, 16),
            (9, 13), (9, 15),
            (11, 13), (11, 15),
            (13, 15),
            (14, 16)
        ]
    },
    {
        "signal_id": "2C042",
        "channels": {
            1: 'Ph1', 2: 'Ph2', 3: 'O1', 4: None,
            5: None, 6: 'Ph6', 7: None, 8: 'Ph8',
            9: None, 10: None, 11: None, 12: None,
            13: 'Ped2', 14: None, 15: None, 16: 'Ped8'
        },
        "removed_diodes": [
            (1, 6),
            (2, 3), (2, 6), (2, 13),
            (3, 6), (3, 13),
            (6, 13),
            (8, 16)
        ]
    }
]

# --- DO NOT CHANGE THE LOGIC BELOW ---
json_path = Path('version_testing/conflict_monitor/conflict_pairs.json')
json_path.parent.mkdir(parents=True, exist_ok=True) # Ensure directory exists
data = json.loads(json_path.read_text()) if json_path.exists() else {}

for signal in signals_data:
    signal_id = signal["signal_id"]
    channels = signal["channels"]
    removed_diodes = signal["removed_diodes"]

    active_channels = [ch for ch, name in channels.items() if name is not None]
    all_pairs = [(min(c1, c2), max(c1, c2)) for i, c1 in enumerate(active_channels) for c2 in active_channels[i+1:]]
    conflict_pairs = [[channels[p[0]], channels[p[1]]] for p in all_pairs if p not in removed_diodes]
    
    data[signal_id] = conflict_pairs
    print(f"Added {len(conflict_pairs)} conflict pairs for signal {signal_id}")

# Custom formatting for single-line pairs
json_str = '{\n'
items = list(data.items())
for i, (sig, pairs) in enumerate(items):
    json_str += f'    "{sig}": [\n'
    json_str += ',\n'.join([f'        ["{p[0]}", "{p[1]}"]' for p in pairs])
    json_str += '\n    ]' + (',' if i < len(items) - 1 else '') + '\n'
json_str += '}\n'

json_path.write_text(json_str)
print("Finished writing to conflict_pairs.json")

Added 52 conflict pairs for signal 12059
Added 24 conflict pairs for signal 13008
Added 70 conflict pairs for signal 13010
Added 31 conflict pairs for signal 2C043
Added 28 conflict pairs for signal 2B049
Added 48 conflict pairs for signal 2B052
Added 40 conflict pairs for signal 2B094
Added 13 conflict pairs for signal 2C042
Finished writing to conflict_pairs.json
